In [34]:
import os
import pandas as pd
import numpy as np
import pyreadstat
from pathlib import Path
from functools import reduce

In [36]:
RAW = Path("data/raw")
OUT = Path("data/processed")
OUT.mkdir(parents=True, exist_ok=True)

In [38]:
# Columns
COLUMNS = {
    "DEMO": ["SEQN","RIDAGEYR","RIAGENDR","RIDRETH3","DMDEDUC2","INDFMPIR"],
    "BMX": ["SEQN","BMXBMI"],
    "BPX": ["SEQN","BPXSY1","BPXDI1","BPXSY2","BPXDI2","BPXSY3","BPXDI3"],
    "GHB": ["SEQN","LBXGH"],
    "GLU": ["SEQN","LBXGLU"],
    "DIQ": ["SEQN","DIQ010","DIQ160","DIQ050","DIQ070"],
}

In [42]:
# Read Data
def read_xpt(path):
    df, meta = pyreadstat.read_xport(path)
    return df

In [46]:
# Load Cycles
def load_cycle(suffix):  # 'D' for 2005-2006 like that D, E, F, G, H, I, J
    parts = {}
    for table, cols in COLUMNS.items():
        fn_list = list(RAW.glob(f"**/{table}_{suffix}.XPT"))
        
        if not fn_list:
            continue
            
        df = read_xpt(fn_list[0])
        
        present_cols = [c for c in cols if c in df.columns]
        
        if not present_cols:
            print(f"Warning: No required columns found in {table}_{suffix}.XPT. Skipping this table.")
            continue
            
        df = df[present_cols]
        parts[table] = df
        
    if not parts:
        return None

    dfs = [v for k,v in parts.items()]
   
    base = reduce(lambda l,r: pd.merge(l,r,on="SEQN", how="outer"), dfs)
    base["cycle"] = suffix
    return base

In [48]:
# Create Labels
def create_label(df):
    # Diabetes definition conditions
    cond_hba1c = df.get("LBXGH").ge(6.5) if "LBXGH" in df else False
    cond_glu = df.get("LBXGLU").ge(126) if "LBXGLU" in df else False
    # DIQ010: ever told you have diabetes (1=yes)
    diq = df.get("DIQ010")
    cond_diq = diq.eq(1) if diq is not None else False
    
    # Combine conditions for the positive label (1.0)
    y = (cond_hba1c | cond_glu | cond_diq).astype("float")
    
    # Assign NaN for people who don't meet the criteria AND have missing data
    y[(~cond_hba1c) & (~cond_glu) & (diq.isna() if diq is not None else True)] = np.nan
    return y

In [50]:
# Creates derived and categorical features
def engineer_features(df):
    # BP averages
    sys_cols = [c for c in df.columns if c.startswith("BPXSY")]
    dia_cols = [c for c in df.columns if c.startswith("BPXDI")]
    if sys_cols:
        df["SBP_mean"] = df[sys_cols].mean(axis=1, skipna=True)
    if dia_cols:
        df["DBP_mean"] = df[dia_cols].mean(axis=1, skipna=True)
        
    # BMI bands
    bins = [0,18.5,25,30,35,100]
    labels = ["Underweight","Normal","Overweight","Obese","VeryObese"]
    if "BMXBMI" in df:
        df["BMI_band"] = pd.cut(df["BMXBMI"], bins=bins, labels=labels, right=False)
        
    # Age bands
    if "RIDAGEYR" in df:
        df["AGE_band"] = pd.cut(df["RIDAGEYR"], bins=[0,18,35,50,65,120], labels=["<18","18-34","35-49","50-64","65+"] , right=False)
        
    return df

In [52]:
def main():
    all_xpt_files = list(RAW.glob("**/*.XPT"))
    
    cycles = sorted({p.stem.split("_")[-1] for p in all_xpt_files if len(p.stem.split("_")) > 1})

    frames = []
    print(f"Found cycles: {cycles}")
    
    for s in cycles:
        print(f"Loading cycle {s}...")
        df = load_cycle(s)
        if df is None or df.empty:
            print(f"Warning: Could not load data for cycle {s}. Skipping.")
            continue
        frames.append(df)
        
    if not frames:
        raise SystemExit("No complete cycle data found in data/raw. Check file names and structure.")
        
   
    full = pd.concat(frames, ignore_index=True)
    
    full["diabetes_label"] = create_label(full)
    full = engineer_features(full)
    
    full = full.dropna(subset=["diabetes_label"]).copy()
    
    keep = ["SEQN","RIDAGEYR","RIAGENDR","RIDRETH3","DMDEDUC2","INDFMPIR","BMXBMI","SBP_mean","DBP_mean","LBXGH","LBXGLU","diabetes_label"]
    
    for k in keep:
        if k not in full.columns:
            full[k] = np.nan
            
    output_path = OUT/"nhanes_diabetes_clean.csv"
    full[keep].to_csv(output_path, index=False)
    print("\n--- Process Complete ---")
    print(f"Total rows in clean dataset: {len(full)}")
    print(f"Saved: {output_path}")

In [54]:
if __name__ == "__main__":
    main()

Found cycles: ['D', 'E', 'F', 'G', 'H', 'I', 'J']
Loading cycle D...
Loading cycle E...
Loading cycle F...
Loading cycle G...
Loading cycle H...
Loading cycle I...
Loading cycle J...

--- Process Complete ---
Total rows in clean dataset: 67201
Saved: data\processed\nhanes_diabetes_clean.csv
